### EDA and discovery


In [2]:
import pandas as pd
import os

data_dir = 'planets-dataset/planet/planet'
df = pd.read_csv(os.path.join(data_dir, 'train_classes.csv'))
df['filename'] = df['image_name'] + '.jpg'

tags_split = df['tags'].str.get_dummies(sep=' ')
df = pd.concat([df, tags_split], axis=1)

print(df.shape)
print(df.columns.duplicated().sum())  # must print 0 before continuing

(40479, 20)
0


In [3]:
import os

print(os.listdir('planets-dataset/planet/planet'))

['sample_submission.csv', 'test-jpg', 'train-jpg', 'train_classes.csv']


In [4]:
#image folder
image_dir = os.path.join(data_dir, 'train-jpg')
print(os.listdir(image_dir)[:5])

['train_0.jpg', 'train_1.jpg', 'train_10.jpg', 'train_100.jpg', 'train_1000.jpg']


In [5]:
tag_counts = tags_split.sum().sort_values(ascending=False)
print(tag_counts)

primary              37513
clear                28431
agriculture          12315
road                  8071
water                 7411
partly_cloudy         7261
cultivation           4477
habitation            3660
haze                  2697
cloudy                2089
bare_ground            862
selective_logging      340
artisinal_mine         339
blooming               332
slash_burn             209
conventional_mine      100
blow_down               98
dtype: int64


In [6]:
cooccurrence = tags_split.T.dot(tags_split)
print(cooccurrence)


                   agriculture  artisinal_mine  bare_ground  blooming  \
agriculture              12315              38          225        32   
artisinal_mine              38             339           40         0   
bare_ground                225              40          862         3   
blooming                    32               0            3       332   
blow_down                   22               0            4         1   
clear                     9150             307          747       311   
cloudy                       0               0            0         0   
conventional_mine           24               4           10         0   
cultivation               3377              18           89        35   
habitation                2737              29          163         4   
haze                       672               5           41         4   
partly_cloudy             2493              27           74        17   
primary                  11972             324     

In [7]:
print(df.columns.tolist())
print(df.columns[df.columns.duplicated()])

['image_name', 'tags', 'filename', 'agriculture', 'artisinal_mine', 'bare_ground', 'blooming', 'blow_down', 'clear', 'cloudy', 'conventional_mine', 'cultivation', 'habitation', 'haze', 'partly_cloudy', 'primary', 'road', 'selective_logging', 'slash_burn', 'water']
Index([], dtype='str')


In [8]:
#tagging priority: resource_extraction > agricultural_clearing > infrastructure_other > undisturbed

import numpy as np

extraction_tags = ['selective_logging', 'artisinal_mine', 'conventional_mine']
agriculture_tags = ['agriculture', 'cultivation', 'slash_burn']
infrastructure_tags = ['road', 'habitation', 'blow_down']

conditions = [
    df[extraction_tags].sum(axis=1) > 0,
    df[agriculture_tags].sum(axis=1) > 0,
    df[infrastructure_tags].sum(axis=1) > 0,
    df['primary'] == 1,
]
choices = ['resource_extraction', 'agricultural_clearing', 'infrastructure_other', 'undisturbed']

df['label'] = np.select(conditions, choices, default='unlabeled')
print(df['label'].value_counts())

label
undisturbed              22038
agricultural_clearing    13300
unlabeled                 2472
infrastructure_other      1900
resource_extraction        769
Name: count, dtype: int64


In [9]:
#remove unlabled rows
df_clean = df[df['label'] != 'unlabeled'].copy()
print(len(df_clean))

38007


In [10]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

classes = np.array(df_clean['label'].unique())
weights = compute_class_weight('balanced', classes=classes, y=df_clean['label'])
class_weight_dict = dict(zip(classes, weights))
print(class_weight_dict)

# convert to tensor in the order your model's output layer expects
class_names = ['undisturbed', 'agricultural_clearing', 'infrastructure_other', 'resource_extraction']
weight_tensor = torch.tensor([class_weight_dict[c] for c in class_names], dtype=torch.float32)

{'undisturbed': np.float64(0.43115300843996734), 'agricultural_clearing': np.float64(0.7144172932330827), 'infrastructure_other': np.float64(5.000921052631579), 'resource_extraction': np.float64(12.355981794538362)}


In [11]:
class_names = ['undisturbed', 'agricultural_clearing', 'infrastructure_other', 'resource_extraction']
weight_tensor = torch.tensor([class_weight_dict[c] for c in class_names], dtype=torch.float32)
print(weight_tensor)

tensor([ 0.4312,  0.7144,  5.0009, 12.3560])


In [12]:
label_to_idx = {name: i for i, name in enumerate(class_names)}
df_clean['label_idx'] = df_clean['label'].map(label_to_idx)

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_clean, 
    test_size=0.2, 
    stratify=df_clean['label'], 
    random_state=42
)
print(train_df['label'].value_counts(normalize=True))
print(test_df['label'].value_counts(normalize=True))

label
undisturbed              0.579839
agricultural_clearing    0.349942
infrastructure_other     0.049992
resource_extraction      0.020227
Name: proportion, dtype: float64
label
undisturbed              0.579847
agricultural_clearing    0.349908
infrastructure_other     0.049987
resource_extraction      0.020258
Name: proportion, dtype: float64


In [13]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [14]:
from torch.utils.data import Dataset
from PIL import Image
import os

class ForestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        label = row['label_idx']
        if self.transform:
            image = self.transform(image)
        return image, label

In [15]:
from torch.utils.data import DataLoader

train_dataset = ForestDataset(train_df, image_dir, transform=train_transform)
test_dataset = ForestDataset(test_df, image_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# quick sanity check
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([32, 3, 128, 128]) torch.Size([32])


In [16]:
import torch.nn as nn
import torch.nn.functional as F

class ForestCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 128 -> 64
        x = self.pool(F.relu(self.conv2(x)))  # 64 -> 32
        x = self.pool(F.relu(self.conv3(x)))  # 32 -> 16
        x = self.pool(F.relu(self.conv4(x)))  # 16 -> 8
        x = x.view(x.size(0), -1)  # flatten
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # raw logits, no softmax here
        return x

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)  # should print 'cuda' on your 1070

model = ForestCNN(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss(weight=weight_tensor.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

cuda


In [18]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.5.1+cu121
12.1
True


In [19]:
import time

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [20]:
import numpy as np
from PIL import Image

def preload_images(df, image_dir, size=128):
    images = np.zeros((len(df), size, size, 3), dtype=np.uint8)
    for i, row in enumerate(df.itertuples()):
        img = Image.open(os.path.join(image_dir, row.filename)).convert('RGB').resize((size, size))
        images[i] = np.array(img)
        if i % 5000 == 0:
            print(f"{i}/{len(df)}")
    return images

train_images = preload_images(train_df, image_dir)
test_images = preload_images(test_df, image_dir)

0/30405
5000/30405
10000/30405
15000/30405
20000/30405
25000/30405
30000/30405
0/7602
5000/7602


In [21]:
start = time.time()
train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
elapsed = time.time() - start

print(f"Epoch time: {elapsed:.1f}s")
print(f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")

Epoch time: 151.2s
Train loss: 1.2205, Train acc: 0.6761


In [22]:
num_epochs = 20
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(num_epochs):
    start = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    elapsed = time.time() - start

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} ({elapsed:.0f}s) — "
          f"train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, "
          f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  → saved new best model (val_acc: {val_acc:.4f})")

Epoch 1/20 (172s) — train_loss: 1.0542, train_acc: 0.7183, val_loss: 0.9483, val_acc: 0.7506
  → saved new best model (val_acc: 0.7506)
Epoch 2/20 (170s) — train_loss: 0.9638, train_acc: 0.7372, val_loss: 0.9218, val_acc: 0.7202
Epoch 3/20 (171s) — train_loss: 0.9344, train_acc: 0.7516, val_loss: 0.9056, val_acc: 0.7833
  → saved new best model (val_acc: 0.7833)
Epoch 4/20 (170s) — train_loss: 0.9063, train_acc: 0.7647, val_loss: 0.9081, val_acc: 0.8057
  → saved new best model (val_acc: 0.8057)
Epoch 5/20 (170s) — train_loss: 0.8712, train_acc: 0.7746, val_loss: 0.8339, val_acc: 0.7674
Epoch 6/20 (171s) — train_loss: 0.8600, train_acc: 0.7727, val_loss: 0.8481, val_acc: 0.8060
  → saved new best model (val_acc: 0.8060)
Epoch 7/20 (171s) — train_loss: 0.8510, train_acc: 0.7843, val_loss: 0.8048, val_acc: 0.8066
  → saved new best model (val_acc: 0.8066)
Epoch 8/20 (171s) — train_loss: 0.8245, train_acc: 0.7923, val_loss: 0.8212, val_acc: 0.7941
Epoch 9/20 (171s) — train_loss: 0.8160, t